# Historical coverage checks

Purpose: determine which comparisons the 2023–2025 data can support. This is coverage screening, not evidence of festive price effects. The revised basket, event windows and eligibility rules are retained. Saved outputs have been cleared. Run top to bottom with the local data; historical scans may take time.

## 1. Setup

This notebook runs independently of Notebook 01. It reads existing local Parquet files and regenerates coverage reports in docs/. Downloads and live availability checks have been removed; keep their existing logs as provenance.

In [ ]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Find the project when Jupyter starts in its root or notebooks folder.
cwd = Path.cwd().resolve()
project_root = next(
    (folder for folder in [cwd, *cwd.parents]
     if (folder / "src" / "check_env.py").is_file()),
    Path(r"C:\Users\User\OneDrive\Desktop\malaysia-food-prices"),
)
assert (project_root / "src" / "check_env.py").is_file(), "Set project_root to your project folder."
raw_dir = project_root / "data" / "raw"
docs_dir = project_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

months_needed = ['2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']
required = ["lookup_item.parquet", "lookup_premise.parquet"] + [
    f"pricecatcher_{month}.parquet" for month in months_needed
]
missing = [name for name in required if not (raw_dir / name).is_file()]
assert not missing, f"Missing local inputs: {missing}"
print("Project:", project_root)
print("Required local files found:", len(required))

## 2. Historical date coverage

Check all 36 months. Missing calendar days remain missing; do not fill them with prices.

In [ ]:
date_checks = []

for month in pd.period_range("2023-01", "2025-12", freq="M"):
    filename = raw_dir / f"pricecatcher_{month}.parquet"

    df = pd.read_parquet(filename, columns=["date"])
    dates = pd.to_datetime(df["date"], errors="raise")

    expected = pd.date_range(
        month.start_time.normalize(),
        month.end_time.normalize(),
    )
    observed = pd.DatetimeIndex(
        dates.dropna().dt.normalize().unique()
    )

    missing = expected.difference(observed)
    outside = observed.difference(expected)

    date_checks.append({
        "month": str(month),
        "rows": len(df),
        "calendar_days": len(expected),
        "observed_days_in_month": len(expected.intersection(observed)),
        "null_dates": int(dates.isna().sum()),
        "missing_days": len(missing),
        "missing_dates": ", ".join(missing.strftime("%Y-%m-%d")),
        "outside_days": len(outside),
        "outside_dates": ", ".join(outside.strftime("%Y-%m-%d")),
    })

date_coverage = pd.DataFrame(date_checks)

output_path = docs_dir / "historical_date_coverage.csv"
date_coverage.to_csv(output_path, index=False)

print(date_coverage[
    ["month", "rows", "calendar_days", "observed_days_in_month",
     "null_dates", "missing_days", "outside_days"]
].to_string(index=False))

print("\nMonths checked:", len(date_coverage))
print("Saved:", output_path)

gaps = date_coverage.loc[
    date_coverage["missing_days"] > 0,
    ["month", "missing_days", "missing_dates"],
]

print(gaps.to_string(index=False))

print(
    "\nTotal calendar dates without records:",
    date_coverage["missing_days"].sum(),
)

## 3. Historical item candidates

Retain the 18-item review to document why some initial choices were replaced.

In [ ]:
review_codes = [
    1, 1111, 992, 918, 1589,
    1593, 129, 1564, 1131, 94,
    114, 103, 1370, 1555, 193,
    2066, 917, 1431,
]

items = pd.read_parquet(raw_dir / "lookup_item.parquet")
basket = pd.DataFrame({"item_code": review_codes})
monthly_results = []

for month in pd.period_range("2023-01", "2025-12", freq="M"):
    print(f"Checking item coverage: {month}", flush=True)

    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet",
        columns=["date", "item_code", "premise_code"],
    )

    monthly["date"] = pd.to_datetime(
        monthly["date"], errors="raise"
    ).dt.normalize()

    source_days = monthly["date"].nunique()

    selected = monthly.loc[
        monthly["item_code"].isin(review_codes)
    ]

    counts = (
        selected.groupby("item_code")
        .agg(
            observations=("item_code", "size"),
            recorded_days=("date", "nunique"),
            premises=("premise_code", "nunique"),
        )
        .reset_index()
    )

    coverage = basket.merge(
        counts,
        on="item_code",
        how="left",
        validate="one_to_one",
    )

    count_columns = ["observations", "recorded_days", "premises"]
    coverage[count_columns] = (
        coverage[count_columns].fillna(0).astype(int)
    )

    coverage["month"] = str(month)
    coverage["source_days"] = source_days
    monthly_results.append(coverage)

    del monthly, selected

historical_coverage = pd.concat(monthly_results, ignore_index=True)

# Summarise consistency across the 36 months.
historical_coverage["has_records"] = (
    historical_coverage["observations"] > 0
)

item_summary = (
    historical_coverage.groupby("item_code")
    .agg(
        months_with_records=("has_records", "sum"),
        total_records=("observations", "sum"),
        min_days_per_month=("recorded_days", "min"),
        median_days_per_month=("recorded_days", "median"),
        min_premises_per_month=("premises", "min"),
    )
    .reset_index()
)

item_summary = basket.merge(
    item_summary,
    on="item_code",
    validate="one_to_one",
).merge(
    items[["item_code", "item", "unit"]],
    on="item_code",
    how="left",
    validate="one_to_one",
)

docs_dir.mkdir(parents=True, exist_ok=True)

historical_coverage.to_csv(
    docs_dir / "historical_item_month_coverage.csv",
    index=False,
)
item_summary.to_csv(
    docs_dir / "historical_item_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nHistorical item summary:")
print(item_summary.to_string(index=False))

print("\nItem-month rows:", len(historical_coverage))
print(
    "Item-month pairs unique:",
    not historical_coverage.duplicated(["item_code", "month"]).any(),
)

focus_codes = [992, 2066, 1593]

for code in focus_codes:
    subset = historical_coverage.loc[
        historical_coverage["item_code"].eq(code)
    ].sort_values("month")

    present = subset.loc[subset["observations"] > 0, "month"]
    absent = subset.loc[subset["observations"].eq(0), "month"]

    print(f"\nItem code: {code}")
    print("First observed month:",
          present.iloc[0] if not present.empty else "None")
    print("Last observed month:",
          present.iloc[-1] if not present.empty else "None")
    print("Months without records:", absent.tolist())

## 4. Rice candidates and revised basket

Compare 10 kg rice products across the whole period, then use the agreed 15 codes for subsequent checks.

In [ ]:
# Select rice products with a listed 10 kg pack size.
rice_mask = (
    items["item"].str.startswith("BERAS", na=False)
    & items["unit"].str.replace(" ", "", regex=False).str.lower().eq("10kg")
)

rice_candidates = items.loc[
    rice_mask, ["item_code", "item", "unit"]
].copy()

rice_results = []

for month in pd.period_range("2023-01", "2025-12", freq="M"):
    print(f"Checking rice: {month}", flush=True)

    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet",
        columns=["date", "item_code", "premise_code"],
    )

    selected = monthly.loc[
        monthly["item_code"].isin(rice_candidates["item_code"])
    ]

    counts = (
        selected.groupby("item_code")
        .agg(
            observations=("item_code", "size"),
            recorded_days=("date", "nunique"),
            premises=("premise_code", "nunique"),
        )
        .reset_index()
    )

    result = rice_candidates[["item_code"]].merge(
        counts, on="item_code", how="left", validate="one_to_one"
    )

    columns = ["observations", "recorded_days", "premises"]
    result[columns] = result[columns].fillna(0).astype(int)
    result["month"] = str(month)
    rice_results.append(result)

    del monthly, selected

rice_history = pd.concat(rice_results, ignore_index=True)
rice_history["has_records"] = rice_history["observations"] > 0

rice_summary = (
    rice_history.groupby("item_code")
    .agg(
        months_with_records=("has_records", "sum"),
        total_records=("observations", "sum"),
        min_days_per_month=("recorded_days", "min"),
        median_days_per_month=("recorded_days", "median"),
        min_premises_per_month=("premises", "min"),
    )
    .reset_index()
    .merge(rice_candidates, on="item_code", validate="one_to_one")
    .sort_values(
        ["months_with_records", "min_days_per_month", "total_records"],
        ascending=False,
    )
)

rice_history.to_csv(
    docs_dir / "historical_rice_month_coverage.csv", index=False
)
rice_summary.to_csv(
    docs_dir / "historical_rice_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nRice coverage summary:")
print(rice_summary.to_string(index=False))

print("\nExpected rows:", len(rice_candidates) * 36)
print("Actual rows:", len(rice_history))
print(
    "Item-month pairs unique:",
    not rice_history.duplicated(["item_code", "month"]).any(),
)

basket_codes = [
    1, 1111, 904, 918, 1589,
    917, 129, 1564, 1131, 94,
    114, 103, 1431, 1555, 193,
]

print("Basket items:", len(basket_codes))
print("Unique codes:", len(set(basket_codes)))
print("Replaced codes absent:",
      not any(code in basket_codes for code in [992, 1593, 1370]))

## 5. State coverage and unmatched premises

Include absent item–state–month combinations and keep unknown locations in an audit.

In [ ]:
premises = pd.read_parquet(raw_dir / "lookup_premise.parquet")
state_lookup = premises[["premise_code", "state"]]

states = sorted(state_lookup["state"].dropna().unique())
months = pd.period_range("2023-01", "2025-12", freq="M")

state_results = []
lookup_checks = []

for month in months:
    print(f"Checking state coverage: {month}", flush=True)

    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet",
        columns=["date", "item_code", "premise_code"],
    )

    selected = monthly.loc[
        monthly["item_code"].isin(basket_codes)
    ].copy()
    del monthly

    selected["date"] = pd.to_datetime(
        selected["date"], errors="raise"
    ).dt.normalize()

    selected = selected.merge(
        state_lookup,
        on="premise_code",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    lookup_checks.append({
        "month": str(month),
        "selected_records": len(selected),
        "unmatched_premise_records": int(
            selected["_merge"].eq("left_only").sum()
        ),
        "records_without_state": int(selected["state"].isna().sum()),
    })

    counts = (
        selected.groupby(["item_code", "state"])
        .agg(
            observations=("item_code", "size"),
            recorded_days=("date", "nunique"),
            premises=("premise_code", "nunique"),
        )
        .reset_index()
    )

    # Include item-state combinations with no observations.
    grid = pd.MultiIndex.from_product(
        [basket_codes, states],
        names=["item_code", "state"],
    ).to_frame(index=False)

    result = grid.merge(
        counts,
        on=["item_code", "state"],
        how="left",
        validate="one_to_one",
    )

    columns = ["observations", "recorded_days", "premises"]
    result[columns] = result[columns].fillna(0).astype(int)
    result["month"] = str(month)
    state_results.append(result)

    del selected

state_coverage = pd.concat(state_results, ignore_index=True)
lookup_audit = pd.DataFrame(lookup_checks)

state_coverage["has_records"] = state_coverage["observations"] > 0

state_summary = (
    state_coverage.groupby(["item_code", "state"])
    ["has_records"].sum()
    .unstack("state")
    .reindex(basket_codes)
    .astype(int)
)

state_coverage.to_csv(
    docs_dir / "historical_item_state_month_coverage.csv",
    index=False,
)
lookup_audit.to_csv(
    docs_dir / "historical_state_lookup_audit.csv",
    index=False,
)

print("\nMonths with observations per item and state (maximum 36):")
print(state_summary.to_string())

print("\nUnmatched premise records:",
      lookup_audit["unmatched_premise_records"].sum())
print("Records without state:",
      lookup_audit["records_without_state"].sum())

print("Expected rows:", len(basket_codes) * len(states) * len(months))
print("Actual rows:", len(state_coverage))
print(
    "Item-state-month combinations unique:",
    not state_coverage.duplicated(
        ["item_code", "state", "month"]
    ).any(),
)

print(
    "Record counts reconcile:",
    state_coverage["observations"].sum()
    + lookup_audit["records_without_state"].sum()
    == lookup_audit["selected_records"].sum(),
)

# Show which months contain records with unmatched premise codes.
problem_months = lookup_audit.loc[
    lookup_audit["unmatched_premise_records"] > 0,
    "month",
].tolist()

unmatched_parts = []

for month in problem_months:
    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet",
        columns=["date", "item_code", "premise_code", "price"],
    )

    mask = (
        monthly["item_code"].isin(basket_codes)
        & ~monthly["premise_code"].isin(premises["premise_code"])
    )

    unmatched = monthly.loc[mask].copy()
    unmatched["source_month"] = month
    unmatched_parts.append(unmatched)

unmatched_records = (
    pd.concat(unmatched_parts, ignore_index=True)
    if unmatched_parts else pd.DataFrame(columns=[
        "date", "item_code", "premise_code", "price", "source_month"
    ])
)

print(unmatched_records.to_string(index=False))
print("\nUnmatched records:", len(unmatched_records))
print(
    "Unmatched premise codes:",
    sorted(unmatched_records["premise_code"].unique().tolist()),
)

unmatched_records.to_csv(
    docs_dir / "historical_unmatched_premise_records.csv",
    index=False,
)

## 6. State coverage depth

Monthly presence alone is insufficient. Compare recorded days and premises. Detailed monthly values remain in historical_item_state_month_coverage.csv.

In [ ]:
state_coverage = pd.read_csv(
    docs_dir / "historical_item_state_month_coverage.csv"
)

state_coverage["has_records"] = (
    state_coverage["observations"] > 0
)

state_depth = (
    state_coverage.groupby(["item_code", "state"])
    .agg(
        months_with_records=("has_records", "sum"),
        total_records=("observations", "sum"),
        min_days=("recorded_days", "min"),
        median_days=("recorded_days", "median"),
        min_premises=("premises", "min"),
        median_premises=("premises", "median"),
    )
    .reset_index()
)

output_path = docs_dir / "historical_item_state_depth.csv"
state_depth.to_csv(output_path, index=False)

# Display the three replacement items and coconut milk,
# which showed substantial geographic gaps.
focus_codes = [904, 917, 1431, 103]

for code in focus_codes:
    print(f"\n--- Item {code} ---")
    print(
        state_depth.loc[state_depth["item_code"].eq(code)]
        .drop(columns="item_code")
        .to_string(index=False)
    )

print("\nSummary rows:", len(state_depth))
print(
    "Item-state pairs unique:",
    not state_depth.duplicated(["item_code", "state"]).any(),
)
print("Saved:", output_path)

## 7. Festival calendar

Preserve the supplied event dates, sources and original verification date. This cell does not reverify external sources.

In [ ]:
docs_dir.mkdir(parents=True, exist_ok=True)

# Government sources checked for the calendar dates.
calendar_2023 = (
    "https://www.kabinet.gov.my/storage/2024/11/hka_2023.pdf"
)
calendar_2024 = (
    "https://www.penang.gov.my/index.php?"
    "Itemid=1904&highlight=WzIwMjRd&id=50&lang=ms"
    "&option=com_sppagebuilder&view=page"
)
calendar_2025 = (
    "https://perak.anm.gov.my/images/JANM/AO/Perak/2024/"
    "GAJI/17102024/TARIKH_EMOLUMEN_2025/"
    "SPANM_Tarikh_dan_Peraturan_Pembayaran_Emolumen_Tahun_2025.pdf"
)

rtm_base = (
    "https://berita.rtm.gov.my/nasional/"
    "senarai-berita-nasional/senarai-artikel/"
)

events = [
    (2023, "chinese_new_year", "2023-01-22", calendar_2023),
    (2023, "ramadan_start", "2023-03-23", calendar_2023),
    (2023, "aidilfitri", "2023-04-22",
     rtm_base + "hari-raya-aidilfitri-disambut-lusa-22-april"),
    (2023, "deepavali", "2023-11-12", calendar_2023),

    (2024, "chinese_new_year", "2024-02-10", calendar_2024),
    (2024, "ramadan_start", "2024-03-12",
     rtm_base + "umat-islam-di-malaysia-mula-berpuasa-selasa-12-mac/"),
    (2024, "aidilfitri", "2024-04-10",
     rtm_base + "hari-raya-aidilfitri-disambut-pada-rabu-10-april/"),
    (2024, "deepavali", "2024-10-31", calendar_2024),

    (2025, "chinese_new_year", "2025-01-29", calendar_2025),
    (2025, "ramadan_start", "2025-03-02",
     rtm_base + "umat-islam-puasa-mulai-ahad-ini/"),
    (2025, "aidilfitri", "2025-03-31",
     rtm_base + "umat-islam-di-malaysia-sambut-aidilfitri-pada-isnin/"),
    (2025, "deepavali", "2025-10-20", calendar_2025),
]

festival_calendar = pd.DataFrame(
    events,
    columns=["year", "event", "anchor_date", "source_url"],
)

festival_calendar["anchor_date"] = pd.to_datetime(
    festival_calendar["anchor_date"], errors="raise"
)
festival_calendar["date_definition"] = (
    festival_calendar["event"]
    .eq("ramadan_start")
    .map({True: "first_fasting_day", False: "first_festival_day"})
)
festival_calendar["verified_on"] = "2026-10-06"

assert len(festival_calendar) == 12
assert not festival_calendar.duplicated(["year", "event"]).any()
assert (
    festival_calendar["anchor_date"].dt.year
    == festival_calendar["year"]
).all()

output_path = docs_dir / "festival_calendar.csv"
festival_calendar.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved = pd.read_csv(output_path)

print(saved[["year", "event", "anchor_date"]].to_string(index=False))
print("\nEvents saved:", len(saved))
print("Events per year:", saved.groupby("year").size().to_dict())
print("Year-event pairs unique:",
      not saved.duplicated(["year", "event"]).any())
print("Saved:", output_path)

## 8. Relative comparison windows

Use each year’s event date. All windows have 14 calendar days. The after window starts at day +2 by design, not because every festival lasts two days.

In [ ]:
calendar = pd.read_csv(
    docs_dir / "festival_calendar.csv",
    parse_dates=["anchor_date"],
)

anchors = calendar.set_index(["year", "event"])["anchor_date"]
window_rows = []

def add_window(year, festival, window, anchor, start_offset, end_offset):
    window_rows.append({
        "year": year,
        "festival": festival,
        "window": window,
        "start_date": anchor + pd.Timedelta(days=start_offset),
        "end_date": anchor + pd.Timedelta(days=end_offset),
    })

for year in sorted(calendar["year"].unique()):
    for festival in ["chinese_new_year", "deepavali"]:
        anchor = anchors.loc[(year, festival)]

        add_window(year, festival, "reference", anchor, -28, -15)
        add_window(year, festival, "lead_up", anchor, -14, -1)
        add_window(year, festival, "after", anchor, 2, 15)

    ramadan = anchors.loc[(year, "ramadan_start")]
    raya = anchors.loc[(year, "aidilfitri")]

    add_window(year, "ramadan_raya", "reference", ramadan, -14, -1)
    add_window(year, "ramadan_raya", "early_ramadan", ramadan, 0, 13)
    add_window(year, "ramadan_raya", "raya_lead_up", raya, -14, -1)
    add_window(year, "ramadan_raya", "after_raya", raya, 2, 15)

windows = pd.DataFrame(window_rows)

windows["calendar_days"] = (
    windows["end_date"] - windows["start_date"]
).dt.days + 1

windows["outside_analysis_period"] = (
    (windows["start_date"] < pd.Timestamp("2023-01-01"))
    | (windows["end_date"] > pd.Timestamp("2025-12-31"))
)

# Check every pair of windows for shared dates.
overlap_rows = []

for first, second in combinations(windows.to_dict("records"), 2):
    start = max(first["start_date"], second["start_date"])
    end = min(first["end_date"], second["end_date"])

    if start <= end:
        overlap_rows.append({
            "window_1": (
                f"{first['year']} {first['festival']} {first['window']}"
            ),
            "window_2": (
                f"{second['year']} {second['festival']} {second['window']}"
            ),
            "overlap_start": start,
            "overlap_end": end,
            "overlap_days": (end - start).days + 1,
        })

overlaps = pd.DataFrame(
    overlap_rows,
    columns=[
        "window_1", "window_2", "overlap_start",
        "overlap_end", "overlap_days",
    ],
)

assert len(windows) == 30
assert windows["calendar_days"].eq(14).all()
assert not windows.duplicated(["year", "festival", "window"]).any()

windows.to_csv(
    docs_dir / "festival_windows_candidate.csv",
    index=False,
    date_format="%Y-%m-%d",
)
overlaps.to_csv(
    docs_dir / "festival_window_overlaps.csv",
    index=False,
    date_format="%Y-%m-%d",
)

print(windows.to_string(index=False))
print("\nWindows:", len(windows))
print("All windows contain 14 days:",
      windows["calendar_days"].eq(14).all())

print("\nOverlapping windows:")
print(overlaps.to_string(index=False) if not overlaps.empty else "None")

print("\nWindows extending outside 2023–2025:")
print(
    windows.loc[windows["outside_analysis_period"]]
    .to_string(index=False)
)

## 9. December 2022 supporting dates

The 2023 Chinese New Year reference window starts in December 2022. This support does not extend the main analysis period.

In [ ]:
destination = raw_dir / "pricecatcher_2022-12.parquet"
df = pd.read_parquet(destination, columns=["date"])
dates = pd.to_datetime(df["date"], errors="raise").dt.normalize()

needed = pd.date_range("2022-12-25", "2022-12-31")
counts = dates.value_counts().reindex(needed, fill_value=0).sort_index()

print("\nRecords on required supporting dates:")
print(counts.to_string())

missing = counts.index[counts.eq(0)]
print("\nRequired dates:", len(needed))
print("Dates with records:", int(counts.gt(0).sum()))
print("Dates without records:",
      missing.strftime("%Y-%m-%d").tolist())

counts.rename_axis("date").rename("records").to_csv(
    docs_dir / "december_2022_supporting_date_coverage.csv"
)

## 10. Initial festive coverage

Require at least four recorded dates in each window and five common premises. These are project screening rules, not statistical significance tests.

In [ ]:
windows = pd.read_csv(
    docs_dir / "festival_windows_candidate.csv",
    parse_dates=["start_date", "end_date"],
)
premises = pd.read_parquet(raw_dir / "lookup_premise.parquet")
states = sorted(premises["state"].dropna().unique())

# Collect observations inside the selected windows, one file at a time.
window_parts = []

needed_months = sorted({
    str(month)
    for row in windows.itertuples()
    for month in pd.period_range(row.start_date, row.end_date, freq="M")
})

for month in needed_months:
    print(f"Checking festive coverage: {month}", flush=True)

    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet",
        columns=["date", "item_code", "premise_code"],
    )

    selected = monthly.loc[
        monthly["item_code"].isin(basket_codes)
    ].copy()
    del monthly

    selected["date"] = pd.to_datetime(
        selected["date"], errors="raise"
    ).dt.normalize()

    selected = selected.merge(
        premises[["premise_code", "state"]],
        on="premise_code",
        how="left",
        validate="many_to_one",
    )

    for row in windows.itertuples():
        mask = selected["date"].between(row.start_date, row.end_date)

        if mask.any():
            part = selected.loc[
                mask, ["date", "item_code", "premise_code", "state"]
            ].copy()
            part["year"] = row.year
            part["festival"] = row.festival
            part["window"] = row.window
            window_parts.append(part)

    del selected

window_records = pd.concat(window_parts, ignore_index=True)

# Unknown states remain in the audit but cannot enter state comparisons.
print(
    "\nWindow records without state:",
    window_records["state"].isna().sum(),
)

keys = ["year", "festival", "window", "item_code", "state"]
grouped = window_records.dropna(subset=["state"]).groupby(keys)

metrics = grouped.agg(
    observations=("item_code", "size"),
    recorded_days=("date", "nunique"),
    premises=("premise_code", "nunique"),
)

premise_sets = grouped["premise_code"].agg(
    lambda values: set(values.dropna())
).to_dict()

# Compare each target window with its festival's reference window.
results = []

for row in windows.loc[windows["window"].ne("reference")].itertuples():
    for code in basket_codes:
        for state in states:
            reference_key = (
                row.year, row.festival, "reference", code, state
            )
            target_key = (
                row.year, row.festival, row.window, code, state
            )

            reference = (
                metrics.loc[reference_key]
                if reference_key in metrics.index
                else {"recorded_days": 0, "premises": 0}
            )
            target = (
                metrics.loc[target_key]
                if target_key in metrics.index
                else {"recorded_days": 0, "premises": 0}
            )

            common = len(
                premise_sets.get(reference_key, set())
                & premise_sets.get(target_key, set())
            )

            reasons = []
            if reference["recorded_days"] < 4:
                reasons.append("reference_dates_below_4")
            if target["recorded_days"] < 4:
                reasons.append("target_dates_below_4")
            if reference["premises"] < 5:
                reasons.append("reference_premises_below_5")
            if target["premises"] < 5:
                reasons.append("target_premises_below_5")
            if common < 5:
                reasons.append("common_premises_below_5")

            results.append({
                "year": row.year,
                "festival": row.festival,
                "target_window": row.window,
                "item_code": code,
                "state": state,
                "reference_days": reference["recorded_days"],
                "target_days": target["recorded_days"],
                "reference_premises": reference["premises"],
                "target_premises": target["premises"],
                "common_premises": common,
                "coverage_pass": not reasons,
                "failure_reasons": "; ".join(reasons),
            })

eligibility = pd.DataFrame(results)

eligibility.to_csv(
    docs_dir / "festival_comparison_coverage.csv",
    index=False,
)

summary = (
    eligibility.groupby(["year", "festival", "target_window"])
    .agg(
        comparisons_checked=("coverage_pass", "size"),
        coverage_passes=("coverage_pass", "sum"),
    )
    .reset_index()
)
summary["coverage_failures"] = (
    summary["comparisons_checked"] - summary["coverage_passes"]
)

print("\nCoverage results:")
print(summary.to_string(index=False))
print("\nTotal comparisons:", len(eligibility))
print(
    "Comparison keys unique:",
    not eligibility.duplicated(
        ["year", "festival", "target_window", "item_code", "state"]
    ).any(),
)

## 11. Check dates among shared premises

Recheck date coverage using premises observed in both windows. Dates are counted collectively across these premises, not separately for each shop.

In [ ]:
# Reuse window_records and eligibility from the previous cell.
keys = ["year", "festival", "window", "item_code", "state"]

# Keep only the columns needed for matched-premise date checks.
group_data = {
    key: group[["premise_code", "date"]].drop_duplicates()
    for key, group in (
        window_records.dropna(subset=["state"]).groupby(keys)
    )
}

matched_checks = []

for row in eligibility.itertuples(index=False):
    reference_key = (
        row.year, row.festival, "reference",
        row.item_code, row.state,
    )
    target_key = (
        row.year, row.festival, row.target_window,
        row.item_code, row.state,
    )

    reference = group_data.get(reference_key)
    target = group_data.get(target_key)

    reference_days = 0
    target_days = 0
    common_count = 0

    if reference is not None and target is not None:
        common = (
            set(reference["premise_code"])
            & set(target["premise_code"])
        )
        common_count = len(common)

        reference_days = reference.loc[
            reference["premise_code"].isin(common), "date"
        ].nunique()

        target_days = target.loc[
            target["premise_code"].isin(common), "date"
        ].nunique()

    matched_checks.append({
        "matched_reference_days": reference_days,
        "matched_target_days": target_days,
        "matched_common_premises": common_count,
    })

matched_eligibility = pd.concat(
    [
        eligibility.reset_index(drop=True),
        pd.DataFrame(matched_checks),
    ],
    axis=1,
)

matched_eligibility["matched_coverage_pass"] = (
    matched_eligibility["coverage_pass"]
    & matched_eligibility["matched_reference_days"].ge(4)
    & matched_eligibility["matched_target_days"].ge(4)
    & matched_eligibility["matched_common_premises"].ge(5)
)

matched_eligibility.to_csv(
    docs_dir / "festival_matched_comparison_coverage.csv",
    index=False,
)

summary = (
    matched_eligibility.groupby(
        ["year", "festival", "target_window"]
    )
    .agg(
        initial_passes=("coverage_pass", "sum"),
        matched_passes=("matched_coverage_pass", "sum"),
    )
    .reset_index()
)

summary["additional_failures"] = (
    summary["initial_passes"] - summary["matched_passes"]
)

print(summary.to_string(index=False))
print("\nComparisons checked:", len(matched_eligibility))
print(
    "Common-premise counts agree:",
    matched_eligibility["matched_common_premises"]
    .eq(matched_eligibility["common_premises"]).all(),
)

## 12. Three-year consistency and failure reasons

Identify comparisons passing in all three years. Failure reasons may overlap; their counts should not be added together.

In [ ]:
# Identify the comparison that failed the stricter check.
new_failures = matched_eligibility.loc[
    matched_eligibility["coverage_pass"]
    & ~matched_eligibility["matched_coverage_pass"],
    [
        "year", "festival", "target_window", "item_code", "state",
        "matched_reference_days", "matched_target_days",
        "matched_common_premises",
    ],
]

print("Additional failure:")
print(new_failures.to_string(index=False))

# Assess each item-state comparison across all three years.
consistency = (
    matched_eligibility.groupby(
        ["festival", "target_window", "item_code", "state"]
    )
    .agg(
        years_checked=("year", "nunique"),
        years_passing=("matched_coverage_pass", "sum"),
    )
    .reset_index()
)

consistency["passes_all_three_years"] = (
    consistency["years_checked"].eq(3)
    & consistency["years_passing"].eq(3)
)

consistency.to_csv(
    docs_dir / "festival_coverage_consistency.csv",
    index=False,
)

# Each value is the number of locations qualifying in all three years.
location_counts = (
    consistency.groupby(["item_code", "festival", "target_window"])
    ["passes_all_three_years"]
    .sum()
    .unstack(["festival", "target_window"])
    .reindex(basket_codes)
)

print("\nLocations passing in all three years (maximum 16):")
print(location_counts.to_string())

print("\nConsistency rows:", len(consistency))
print("Every comparison checked across three years:",
      consistency["years_checked"].eq(3).all())

checks = matched_eligibility.assign(
    insufficient_dates=(
        matched_eligibility["matched_reference_days"].lt(4)
        | matched_eligibility["matched_target_days"].lt(4)
    ),
    insufficient_common_premises=(
        matched_eligibility["matched_common_premises"].lt(5)
    ),
)

failure_summary = (
    checks.groupby("item_code")
    .agg(
        comparisons=("matched_coverage_pass", "size"),
        passed=("matched_coverage_pass", "sum"),
        insufficient_dates=("insufficient_dates", "sum"),
        insufficient_common_premises=(
            "insufficient_common_premises", "sum"
        ),
    )
    .reindex(basket_codes)
)

print(failure_summary.to_string())

output_path = docs_dir / "festival_coverage_failure_summary.csv"
failure_summary.to_csv(output_path, index=True)

saved = pd.read_csv(output_path)

print("Saved:", output_path)
print("Items:", len(saved))
print("Total comparisons:", saved["comparisons"].sum())
print("Total passing:", saved["passed"].sum())

## 13. Retailer labels

Strip surrounding whitespace in a separate column; preserve the source label and missing values.

In [ ]:
premises = pd.read_parquet(raw_dir / "lookup_premise.parquet")

type_counts = (
    premises.groupby("premise_type", dropna=False)
    ["premise_code"]
    .nunique()
    .sort_values(ascending=False)
)

print("Distinct premises per retailer type in the lookup:")
print(type_counts.to_string())

print("\nKnown retailer types:", premises["premise_type"].nunique())
print("Premises without a retailer type:",
      premises["premise_type"].isna().sum())

# Preserve the original label; normalise a separate working column.
premises["premise_type_clean"] = (
    premises["premise_type"].str.strip()
)

changed = (
    premises["premise_type"].notna()
    & premises["premise_type"].ne(premises["premise_type_clean"])
)

print("Labels changed:", changed.sum())
print("Categories before:", premises["premise_type"].nunique())
print("Categories after:", premises["premise_type_clean"].nunique())

print(
    "Wet-market premises:",
    premises.loc[
        premises["premise_type_clean"].eq("Pasar Basah"),
        "premise_code",
    ].nunique(),
)

print("Missing types preserved:",
      premises["premise_type_clean"].isna().sum())

## 14. Retailer coverage and depth

Compare the same item and unit. National monthly presence does not establish adequate state-level or festive-window coverage.

In [ ]:
premises = pd.read_parquet(raw_dir / "lookup_premise.parquet")
premises["premise_type_clean"] = premises["premise_type"].str.strip()

retailer_types = sorted(
    premises["premise_type_clean"].dropna().unique()
)

basket_codes = [
    1, 1111, 904, 918, 1589,
    917, 129, 1564, 1131, 94,
    114, 103, 1431, 1555, 193,
]

grid = pd.MultiIndex.from_product(
    [basket_codes, retailer_types],
    names=["item_code", "premise_type_clean"],
).to_frame(index=False)

results = []
audit_rows = []

for month in pd.period_range("2023-01", "2025-12", freq="M"):
    print(f"Checking retailer coverage: {month}", flush=True)

    monthly = pd.read_parquet(
        raw_dir / f"pricecatcher_{month}.parquet",
        columns=["date", "item_code", "premise_code"],
    )

    selected = monthly.loc[
        monthly["item_code"].isin(basket_codes)
    ].copy()
    del monthly

    selected["date"] = pd.to_datetime(
        selected["date"], errors="raise"
    ).dt.normalize()

    selected = selected.merge(
        premises[["premise_code", "premise_type_clean"]],
        on="premise_code",
        how="left",
        validate="many_to_one",
    )

    audit_rows.append({
        "month": str(month),
        "selected_records": len(selected),
        "records_without_type": int(
            selected["premise_type_clean"].isna().sum()
        ),
    })

    counts = (
        selected.groupby(["item_code", "premise_type_clean"])
        .agg(
            observations=("item_code", "size"),
            recorded_days=("date", "nunique"),
            premises=("premise_code", "nunique"),
        )
        .reset_index()
    )

    coverage = grid.merge(
        counts,
        on=["item_code", "premise_type_clean"],
        how="left",
        validate="one_to_one",
    )

    columns = ["observations", "recorded_days", "premises"]
    coverage[columns] = coverage[columns].fillna(0).astype(int)
    coverage["month"] = str(month)
    results.append(coverage)

    del selected

retailer_coverage = pd.concat(results, ignore_index=True)
retailer_audit = pd.DataFrame(audit_rows)

retailer_coverage["has_records"] = (
    retailer_coverage["observations"] > 0
)

retailer_summary = (
    retailer_coverage.groupby(["item_code", "premise_type_clean"])
    .agg(
        months_with_records=("has_records", "sum"),
        total_records=("observations", "sum"),
        min_days=("recorded_days", "min"),
        median_days=("recorded_days", "median"),
        min_premises=("premises", "min"),
        median_premises=("premises", "median"),
    )
    .reset_index()
)

retailer_coverage.to_csv(
    docs_dir / "historical_item_retailer_month_coverage.csv",
    index=False,
)
retailer_summary.to_csv(
    docs_dir / "historical_item_retailer_summary.csv",
    index=False,
)
retailer_audit.to_csv(
    docs_dir / "historical_retailer_lookup_audit.csv",
    index=False,
)

presence = retailer_summary.pivot(
    index="item_code",
    columns="premise_type_clean",
    values="months_with_records",
).reindex(basket_codes)

print("\nMonths with observations by retailer type (maximum 36):")
print(presence.to_string())

print("\nExpected rows:", 15 * len(retailer_types) * 36)
print("Actual rows:", len(retailer_coverage))
print(
    "Item-type-month combinations unique:",
    not retailer_coverage.duplicated(
        ["item_code", "premise_type_clean", "month"]
    ).any(),
)
print(
    "Records without retailer type:",
    retailer_audit["records_without_type"].sum(),
)
print(
    "Record counts reconcile:",
    retailer_coverage["observations"].sum()
    + retailer_audit["records_without_type"].sum()
    == retailer_audit["selected_records"].sum(),
)

main_types = [
    "Hypermarket",
    "Kedai Runcit",
    "Pasar Basah",
    "Pasar Mini",
    "Pasar Raya / Supermarket",
]

for code in [1, 1431]:
    print(f"\n--- Item {code}: retailer coverage depth ---")

    subset = retailer_summary.loc[
        retailer_summary["item_code"].eq(code)
        & retailer_summary["premise_type_clean"].isin(main_types)
    ]

    print(subset.to_string(index=False))